# Pytorch의 nn.Embedding
- Pytorch의 Embedding Layer는 word2vec과 마찬가지로 word embedding vector를 찾는 **Lookup Table**이다.
    - 단어의 **정수의 고유 index**가 입력으로 들어오면 Embedding Layer의 **그 index의 Vector**를 출력한다.
    - 모델이 학습되는 동안 모델이 풀려는 문제에 맞는 값으로 Embedding Layer의 vector들이 업데이트 된다.
    - Word2Vec의 embedding vector 학습을 nn.Embedding은 자신이 포함된 모델을 학습 하는 과정에서 한다고 생각하면 된다.

In [ ]:
import torch
import torch.nn as nn

embedding_model = nn.Embedding(
    num_embeddings=20000,
    embedding_dim=10,
    padding_idx=0,      
)


In [ ]:
weight = embedding_model.weight
weight.shape
weight[0]

In [ ]:
# 문장을 tokenizer를 통해 토큰화 한 결과.
# ""나는"-30, "어제"-100, "밥을"-600, "먹었다"-7200 "."- 5"
doc_token_ids = torch.tensor([[30, 100, 600, 7200, 5]], dtype=torch.int64)  # 한문장
# doc_token_ids = torch.tensor([[30, 100, 600, 7200, 5],[30, 100, 600, 7200, 5],[30, 100, 600, 7200, 5]], dtype=torch.int64)  # 여러문장

doc_embedding_vector = embedding_model(doc_token_ids)

doc_embedding_vector.shape

In [ ]:
doc_embedding_vector[0][1]

# 네이버 영화 댓글 감성분석(Sentiment Analysis)

## 감성분석(Sentiment Analysis) 이란
입력된 텍스트가 **긍적적인 글**인지 **부정적인**인지 또는 **중립적인** 글인지 분석하는 것을 감성(감정) 분석이라고 한다.   
이를 통해 기업이 고객이 자신들의 기업 또는 제품에 대해 어떤 의견을 가지고 있는지 분석한다.   
자연어 생성과 이해중 이해 관련 분야이다.

# Dataset, DataLoader 생성

## Korpora에서 Naver 영화 댓글 dataset 가져오기
- https://github.com/ko-nlp/Korpora
- http://github.com/e9t/nsmc/
    - input: 영화댓글
    - output: 0(부정적댓글), 1(긍정적댓글)
### API
- **corpus 가져오기**
    - `Korpora.load('nsmc')`
- **text/label 조회**
    - `corpus.get_all_texts()` : 전체 corpus의 text들을 tuple로 반환
    - `corpus.get_all_labels()`: 전체 corpus의 label들을 list로 반환
- **train/test set 나눠서 조회**
    - `corpus.train`
    - `corpus.test`
    - `LabeledSentenceKorpusData` 객체에 text와 label들을 담아서 제공.
        - `LabeledSentenceKorpusData.texts`: text들 tuple로 반환.
        - `LabeledSentenceKorpusData.labels`: label들 list로 반환.

## 데이터 로딩

In [ ]:
from Korpora import Korpora
corpus = Korpora.load('nsmc')

In [ ]:
all_inputs = corpus.get_all_texts()
all_labels = corpus.get_all_labels()

In [ ]:
all_inputs[:5]

In [ ]:
all_labels[:5]

In [ ]:
len(all_inputs)

In [ ]:
print(type(corpus.train))
corpus.train

In [ ]:
corpus.train.texts[:5]
corpus.train.labels[:5]

In [ ]:
corpus.test

## 토큰화
1. 형태소 단위 token화
    - konlpy로 token화 한 뒤 다시 한 문장으로 만든다.
2. 1에서 처리한 corpus를 BPE 로 token화
   
### 전처리 함수

#### 형태소 단위 분절

In [ ]:
import string
string.punctuation

In [ ]:
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text)
    text = [token.lemma for token in kiwi.tokenize(text)]
    return ' '.join(text)

In [ ]:
print(all_inputs[100])
text_preprocessing(all_inputs[100])

In [ ]:
train_texts = corpus.train.texts
train_inputs = [text_preprocessing(txt) for txt in train_texts]

test_texts = corpus.test.texts
test_inputs = [text_preprocessing(txt) for txt in test_texts]

train_labels = corpus.train.labels
test_labels = corpus.test.labels

In [ ]:
import os
os.makedirs('data/nsmc')

train_data = {"text": train_inputs, "label": train_labels}
test_data = {"text": test_inputs, "label": test_labels}

import pickle
with open("data/nsmc/preprocessing_train.pkl", "wb") as fo:
    pickle.dump(train_data, fo)

with open("data/nsmc/preprocessing_test.pkl", "wb") as fo:
    pickle.dump(test_data, fo)

In [ ]:
import pickle
with open("data/nsmc/preprocessing_train.pkl", "rb") as fi:
    train = pickle.load(fi)

with open("data/nsmc/preprocessing_test.pkl", "rb") as fi:
    test = pickle.load(fi)

train_inputs = train['text']
train_labels = train['label']

test_inputs = test['text']
test_labels = test['label']

In [ ]:
all_inputs = train_inputs + test_inputs

### 토큰화
- Subword 방식 토큰화 적용
- Byte Pair Encoding 방식으로 huggingface tokenizer 사용
    - BPE: 토큰을 글자 단위로 나눈뒤 가장 자주 등장하는 글자 쌍(byte paire)를 찾아 합친뒤 어휘사전에 추가한다.
    - https://huggingface.co/docs/tokenizers/quicktour
    - `pip install tokenizers`

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size = 30_000

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=5,
    special_tokens=["<pad>","<unk>"],
    continuing_subword_prefix="##"
)

tokenizer.train_from_iterator(all_inputs, trainer=trainer)

In [ ]:
tokenizer.get_vocab_size()

In [ ]:
# 저장
tokenizer.save("saved_models/nsmc_bpe_tokenizer.json")
# 불러오기 : load_tokenizer = Tokenizer.from_file('경로')

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("saved_models/nsmc_bpe_tokenizer.json")

In [ ]:
# 인코딩 테스트
idx = 100
print(all_inputs[idx])

encode = tokenizer.encode(all_inputs[idx])

print(encode.tokens)
print(encode.ids)

In [ ]:
tokenizer.decode(encode.ids)

In [ ]:
print(encode.ids)

## Dataset, DataLoader 생성

In [ ]:
train_inputs[5]

In [ ]:
tokenizer.encode(train_inputs[5]).ids

In [ ]:
tokenizer.token_to_id("<pad>")

In [ ]:
# 사용자 정의 Dataset 생성

import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - 댓글 리스트. 리스트에 댓글들을 담아서 받는다. ["댓글", "댓글", ...]
        labels: list - Label 리스트. (댓글의 긍부정 여부 - 긍정: 1, 부정: 0)
        max_length: 개별 댓글의 token 최대 개수. 모든 댓글의 토큰수를 max_length에 맞춘다.
        tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.labels = labels
        self.texts = [
            self.__pad_token_sequences(tokenizer.encode(txt).ids) for txt in texts
        ]    

    ###########################################################################################
    # id로 구성된 개별 문장 token list를 받아서 패딩 추가 [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    ############################################################################################
    def __pad_token_sequences(self, token_sequences):
        """
        token id로 구성된 개별 문서(댓글)의 token_id list를 받아서 max_length 길이에 맞추는 메소드
        max_length 보다 토큰수가 적으면 <pad> 토큰 추가, 많으면 max_length 크기로 줄인다.
            ex) max_length=5 이고 pad토큰 id가 0이라면
                [20, 2, 1] => [20, 2, 1, 0, 0]
                [20, 21, 30, 34, 60, 17, 21, 33] -> [20, 21, 30, 34, 60]
        """
    
        pad_token_id = self.tokenizer.token_to_id("<pad>")
    
        seq_len = len(token_sequences)
    
        if self.max_length < seq_len:
            result = token_sequences[:self.max_length]
        else:
            result = token_sequences + ([pad_token_id] * (self.max_length - seq_len))

        return result
        
    def __len__(self):
        
        return len(self.texts)


    def __getitem__(self, idx):
        """
        idx 번째 text와 label을 학습 가능한 type으로 변환해서 반환
        Parameter
            idx: int 조회할 index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) - 댓글 토큰_id 리스트, 정답 Label
        """
        comment = torch.tensor(self.texts[idx], dtype=torch.int64)
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
    
        return comment, label

In [ ]:
max_length = 30
trainset = NSMCDataset(train_inputs, train_labels, max_length, tokenizer)
testset = NSMCDataset(test_inputs, test_labels, max_length, tokenizer)

len(trainset), len(testset)

In [ ]:
trainset[110]


In [ ]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=64)

In [ ]:
len(train_loader), len(test_loader)

# 모델링
- Embedding Layer를 이용해 Word Embedding Vector를 추출한다.
- LSTM을 이용해 Feature 추출
- Linear + Sigmoid로 댓글 긍정일 확률 출력
  
![outline](figures/rnn/RNN_outline.png)

## 모델 정의

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import torch
import torch.nn as nn

class NSMCClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers=1, bidirectional=True, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=dropout
        )

        self.classifier = nn.Linear(
            in_features= hidden_size * 2 if bidirectional else hidden_size,
            out_features=1
        )

        self.sigmoid = nn.Sigmoid()


    def forward(self, X):
        """
        Args: 
            X(torch.Tensor): 입력 문서의 토큰 리스트. shape: [batch_size, seq_length(max_length)], [64, 30]
            연산 순서 -> embedding_model =>(transpose)=>lstm => classifier => sigmoid
        """
        embedding_vector = self.embedding(X)
        
        embedding_vector = embedding_vector.transpose(1, 0)

        out, _ = self.lstm(embedding_vector)

        output = self.classifier(out[-1])
        last_output = self.sigmoid(output)
        
        return last_output


## 모델 생성

In [ ]:
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 100
hidden_size = 64
num_layers = 2
bidirectional = True
dropout = 0.3

In [ ]:
# torch info로 모델 확인
from torchinfo import summary
dummy_data = torch.randint(1, 10, (64, max_length))
summary(
    NSMCClassifier(vocab_size, embedding_dim, hidden_size, num_layers, bidirectional, dropout),
    input_data=dummy_data
)

## 학습

### Train/Test 함수 정의

In [ ]:
# 1 에폭 학습 함수.
def train(model, dataloader, loss_fn, optimizer, device="cpu"):
    # 1. 모델을 train 모드로 변경
    model.train()
    # 2. 모델을 device로 이동
    model = model.to(device)

    # 1 에폭 학습
    train_loss = 0.0
    for X, y in dataloader:
        # 1 step 학습
        # 1. X, y를 device 이동
        X, y = X.to(device), y.to(device)

        # 2. 추론
        pred = model(X)

        # 3. loss 계산
        loss = loss_fn(pred, y)

        # 4. gradient 값 계산 - 오차역전파
        loss.backward()

        # 5. weight/bias(파라미터) 업데이트 = new_weight = weight.data - weight.grad * 학습율
        optimizer.step()

        # 6. grad 초기화
        optimizer.zero_grad()

        # loss값 누적
        train_loss += loss.item()

    return train_loss / len(dataloader) # 1 에폭 학습 loss를 반환.

In [ ]:
@torch.no_grad
def eval(model, dataloader, loss_fn, device="cpu"):
    """모델 평가/검증 함수"""
    # 모델을 eval 모드로 변경. (평가/추론)
    model.eval()
    model.to(device)

    eval_loss, eval_acc = 0.0, 0.0
    
    for X,  y in dataloader:
        # 1. X, y를 device이동
        X, y = X.to(device), y.to(device)

        # 2. 추론  - 양성일 확률이 출력.
        pred_proba = model(X)
        pred_label = (pred_proba > 0.5).type(torch.int32)

        # 3. 평가(loss, accuracy)
        eval_loss += loss_fn(pred_proba, y).item()
        eval_acc += (pred_label == y).sum().item()  # 현재 step에서 몇개 맞았는지 대입.

    return eval_loss / len(dataloader), eval_acc / len(dataloader.dataset)

### Train

In [ ]:
lr = 0.0001
epochs = 3

model = NSMCClassifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    dropout=dropout
).to(device)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
from time import time
s = time()

train_loss_list = []
eval_loss_list = []
eval_acc_list = []

for epoch in range(epochs):
    train_loss = train(model, train_loader, loss_fn, optimizer, device)
    eval_loss, eval_acc = eval(model, test_loader, loss_fn, device)
    train_loss_list.append(train_loss)
    eval_loss_list.append(eval_loss)
    eval_acc_list.append(eval_acc)
    print(train_loss, eval_loss, eval_acc, sep=" || ")

e = time()
print("학습에 걸린 시간:", (e-s), "초")

## 모델저장

In [ ]:
torch.save(model, "saved_models/nsmc_lstm_model.pt")

In [ ]:
load_model = torch.load("saved_models/nsmc_lstm_model.pt", weights_only=False)

# 서비스

## 전처리 함수들

In [ ]:
# from konlpy.tag import Okt
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text)
    text = [token.lemma for token in kiwi.tokenize(text)]
    return ' '.join(text)

In [ ]:
def pad_token_sequences(token_sequences, max_length):
    """padding 처리 메소드."""
    pad_token = tokenizer.token_to_id('<pad>')  
    seq_length = len(token_sequences)           
    result = None
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [ ]:
def predict_data_preprocessing(text_list):
    """
    모델에 입력할 수 있는 input data를 생성
    Parameter:
        text_list: list - 추론할 댓글리스트
    Return
        torch.LongTensor - 댓글 token_id tensor
    """
    # 기본 전처리
    text_list = [text_preprocessing(txt) for txt in text_list]
    # 토큰화
    token_list = [tokenizer.encode(txt).ids for txt in text_list]
    # 토큰개수 max_length에 맞추기
    token_list = [pad_token_sequences(token, max_length) for token in token_list]
    
    return torch.tensor(token_list, dtype=torch.int64)

## 추론

In [ ]:
comment_list = ["아 진짜 재미없다.", "여기 식당 먹을만 해요", "이걸 영화라고 만들었냐?", "기대 안하고 봐서 그런지 괜찮은데.", "이걸 영화라고 만들었나?", "아! 뭐야 진짜.", "재미있는데.", "연기 짱 좋아. 한번 더 볼 의향도 있다.", "뭐 그럭저럭"]
input_tensor = predict_data_preprocessing(comment_list)
input_tensor.shape

In [ ]:
input_tensor[1]

In [ ]:
with torch.no_grad():
    pred_proba = model(input_tensor)
    pred_label = (pred_proba > 0.5).type(torch.int32)
    for txt, pred_label in zip(comment_list, pred_label):
        print(txt, end=" ||||||| ")
        print("긍정적 댓글" if pred_label.item()==1 else "부정적 댓글")

In [ ]:
pred_proba


In [ ]:
pred_label